In [0]:
from pyspark.sql.functions import *

In [0]:
df_bronze_visit = spark.readStream.table('healthcare.bronze.visit')
silver_table = 'healthcare.silver.visit'
checkpoint_path = '/Volumes/healthcare/silver/checkpoint/silver_visit_files/v1' 

In [0]:
df_bronze_visit = df_bronze_visit.dropDuplicates(['visit_id']).withColumn('updated_at', current_timestamp())

In [0]:
def merge_visit_data(batch_df,batch_id):
    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format('delta').mode('append').saveAsTable(silver_table)
        return
    
    from delta.tables import DeltaTable

    final_table = DeltaTable.forName(spark,silver_table)
    (
        final_table.alias('Target')
            .merge(
            batch_df.alias('source'),
            'Target.visit_id = source.visit_id'
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


(
    df_bronze_visit.writeStream
        .format('delta')
        .foreachBatch(merge_visit_data)
        .option('checkpointLocation',checkpoint_path)
        .trigger(availableNow = True)
        .start()
)
